<a href="https://colab.research.google.com/github/dnceight/Unicafam_Modelos_Analiticos_Indra/blob/main/Sesion5_Regresion_Lineal_Logistica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

DATOS

In [1]:
import pandas as pd
import numpy as np


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

np.random.seed(42)
n = 400
casas = pd.DataFrame({
    "area_m2": np.random.uniform(35, 160, n).round(),
    "habitaciones": np.random.randint(1, 5, n),
    "estrato": np.random.randint(1, 7, n)})
casas["precio_M"] = (40 + 2.8 * casas["area_m2"] + 15 * casas["habitaciones"]
                     + 55 * casas["estrato"] + np.random.normal(0, 40, n)).round()
print(casas.describe().round(1).loc[["mean", "min", "max"]])


      area_m2  habitaciones  estrato  precio_M
mean     96.8           2.4      3.6     548.3
min      36.0           1.0      1.0     182.0
max     159.0           4.0      6.0     983.0


ENTRENAR LA REGRESIÓN LINEAL

In [3]:
X = casas[["area_m2", "habitaciones", "estrato"]]
y = casas["precio_M"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42)

modelo = LinearRegression()
modelo.fit(X_train, y_train)

print("Intercepto:", round(modelo.intercept_, 1))
for var, coef in zip(X.columns, modelo.coef_):
    print(f"  {var:13} {coef:7.2f}")


Intercepto: 51.1
  area_m2          2.76
  habitaciones    15.81
  estrato         53.98


EVALUAR Y PREDECIR

In [4]:
pred = modelo.predict(X_test)
mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)
base = mean_absolute_error(y_test, [y_train.mean()] * len(y_test))
print(f"MAE modelo: {mae:.1f} M | MAE ingenuo: {base:.1f} M")
print(f"R² en prueba: {r2:.3f}")

nueva = pd.DataFrame({"area_m2": [72], "habitaciones": [3],
                      "estrato": [4]})
print(f"Precio estimado: {modelo.predict(nueva)[0]:.0f} millones")

MAE modelo: 33.4 M | MAE ingenuo: 123.4 M
R² en prueba: 0.923
Precio estimado: 513 millones


CHURN

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report

np.random.seed(7)
n = 1000
cli = pd.DataFrame({
    "meses_cliente": np.random.randint(1, 72, n),
    "quejas_90d": np.random.poisson(1, n),
    "pago_mensual": np.random.uniform(30, 150, n).round(),
    "plan_prepago": np.random.binomial(1, 0.4, n)})
z = (-1 - 0.05 * cli["meses_cliente"] + 0.9 * cli["quejas_90d"]
     + 0.01 * cli["pago_mensual"] + 0.8 * cli["plan_prepago"])
cli["fuga"] = np.random.binomial(1, 1 / (1 + np.exp(-z)))
print(cli["fuga"].value_counts(normalize=True).round(2).to_dict())


{0: 0.62, 1: 0.38}
